# 05 Experiments Over Baseline 2
In this next notebook, I will take what I've learned from my previous experiments and try a chunking experiment with the FinBERT model.

I will also be testing my hypothesis that the reason my models don't learn well from the earnings call transcripts is because the text in there are generally all positive anyway. Therefore, a sentiment classifier (such as FinBERT) won't be a good way to gauge the text.

# Imports

In [ ]:
!pip install gensim nltk evaluate transformers datasets

!pip install -q -U bitsandbytes
# !pip install -q -U bitsandbytes flash_attn
!pip install flash_attn==2.7.4.post1 --no-build-isolation

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.9/72.9 MB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 53.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 56.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 40.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 827.7 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 87.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.0/6.0 MB 39.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  C

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import date
from tqdm import tqdm
import os
import warnings
from pprint import pprint

import sklearn
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, precision_recall_fscore_support

In [ ]:
from gensim.models import Word2Vec
import nltk
from nltk.util import ngrams
from nltk.tokenize import word_tokenize
from collections import defaultdict, Counter
import evaluate

import tensorflow as tf
from keras.models import Model
from keras.layers import Input, Embedding, Lambda, Dense
from tensorflow.keras.layers import TextVectorization
from tensorflow.keras.preprocessing.text import Tokenizer
from keras.preprocessing.sequence import pad_sequences

import torch
import torch.nn as nn

from datasets import Dataset, load_dataset

from transformers import BertTokenizer, BertModel, BertPreTrainedModel, BertForSequenceClassification
from transformers import AutoTokenizer, AutoModel, AutoModelForSequenceClassification, BertConfig
from transformers import TrainingArguments, Trainer, EarlyStoppingCallback
from transformers import pipeline, BitsAndBytesConfig, AutoModelForCausalLM

nltk.download('punkt')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


True

## Loading in full dataset

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Changing directory to get the data
# data_path = '/content/drive/MyDrive/MIDS DATASCI 266/MIDS DATASCI 266 Final Project'
data_path = '/content/drive/MyDrive/Work Stuff/Berkeley MIDS Stuff/Berkeley MIDS 266 Stuff/w266_project'
os.chdir(data_path)
os.getcwd()

'/content/drive/MyDrive/Work Stuff/Berkeley MIDS Stuff/Berkeley MIDS 266 Stuff/w266_project'

In [ ]:
# Retrieving data
full_data_path = 'SNP500_Transcripts_Price_2015_to_2024.csv'
full_data = pd.read_csv(full_data_path, sep='|', index_col=0)

In [ ]:
# Limiting the data to only the text and the chosen target variable
model_df = full_data[['Text', 'Close_5_dir']].copy()
model_df['Close_5_dir'] = model_df['Close_5_dir'].astype(np.int64)

model_df

,Text,Close_5_dir
0,"Good afternoon. My name is Karen, and I'll be ...",0
1,"Ladies and gentlemen, thank you for standing b...",1
2,"Good day, ladies and gentlemen, and welcome to...",0
3,"Good morning, ladies and gentlemen, and welcom...",1
4,"Good morning, ladies and gentlemen, and welcom...",1
...,...,...
17228,"Good day, and thank you for standing by. Welco...",0
17229,Welcome to Lennar's Fourth Quarter Earnings Co...,0
17230,"Good afternoon, everyone. Welcome to NIKE, Inc...",0
17231,"Good day, and welcome to the FedEx Fiscal Year...",0


# Running chunked FinBERT on whole transcript and whole dataset
As we saw in our truncated FinBERT test, using the first chunk of tokens from an earnings transcript doesn't generate great results, and excessive learning by unfreezing transformer layers cause the validation loss to skyrocket. Therefore, let's fine-tune a model that uses the whole transcript - via a chunking strategy - to see if we can improve our predictions.

Note*: This wasn't possible in the beginning since we did not have the compute units or GPU to process all this data (and also in a timely manner).

## Loading in FinBERT
We'll first load in the FinBERT AutoModel for use later.

In [ ]:
# Loading the FinBERT model (without training) and tokenizer from Hugging face
finbert_model_name = "ProsusAI/finbert"
finbert_tokenizer = AutoTokenizer.from_pretrained(finbert_model_name)
finbert = AutoModel.from_pretrained(finbert_model_name)
finbert.eval()

# Getting the device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# Checking out the parameters
for name, param in finbert.named_parameters():
    print(name, param.shape)

embeddings.word_embeddings.weight torch.Size([30522, 768])
embeddings.position_embeddings.weight torch.Size([512, 768])
embeddings.token_type_embeddings.weight torch.Size([2, 768])
embeddings.LayerNorm.weight torch.Size([768])
embeddings.LayerNorm.bias torch.Size([768])
encoder.layer.0.attention.self.query.weight torch.Size([768, 768])
encoder.layer.0.attention.self.query.bias torch.Size([768])
encoder.layer.0.attention.self.key.weight torch.Size([768, 768])
encoder.layer.0.attention.self.key.bias torch.Size([768])
encoder.layer.0.attention.self.value.weight torch.Size([768, 768])
encoder.layer.0.attention.self.value.bias torch.Size([768])
encoder.layer.0.attention.output.dense.weight torch.Size([768, 768])
encoder.layer.0.attention.output.dense.bias torch.Size([768])
encoder.layer.0.attention.output.LayerNorm.weight torch.Size([768])
encoder.layer.0.attention.output.LayerNorm.bias torch.Size([768])
encoder.layer.0.intermediate.dense.weight torch.Size([3072, 768])
encoder.layer.0.inter

## Building and testing model
Let's build and test our new model here. Specifically, we just want to make sure everything works, particularly with a smaller dataset.

### Tokenizing and chunking
I'll first define a function takes the earnings call text from a Huggingface dataset and splits it into 512-token chunks after tokenization, cutting off at 8192 tokens if needed and padding it if it's shorter.

In [ ]:
# Defining max total tokens and chunk size
MAX_TOTAL_TOKENS = 8192
CHUNK_SIZE = 512

# Defining function to tokenize and chunk the batched text
def batched_tokenize_and_chunk(batch, tokenizer, chunk_size=CHUNK_SIZE, max_total_tokens=MAX_TOTAL_TOKENS):

    # Initializing input_ids and attention_mask batch lists
    input_ids_batch = []
    attention_mask_batch = []

    # Going through each text
    for text in batch['text']:

        # Tokenizing and truncating/padding
        tokens = tokenizer.encode(
            text,
            padding="max_length",
            truncation=True,
            max_length=max_total_tokens
        )
        # Breaking into chunks of 512 tokens and getting a list for the ids and masks
        chunks = [tokens[i:i + chunk_size] for i in range(0, len(tokens), chunk_size)]
        masks = [[1 if token != tokenizer.pad_token_id else 0 for token in chunk] for chunk in chunks]
        input_ids_batch.append(torch.tensor(chunks))
        attention_mask_batch.append(torch.tensor(masks))

    # Getting the output
    output = {
        "input_ids": input_ids_batch,
        "attention_mask": attention_mask_batch,
    }

    # Adding the label (if exists)
    if "label" in batch:
        output["labels"] = torch.tensor(batch["label"])

    return output

# Defining follow-up function to collate everything into torch tensors
def chunked_collate_fn(batch):

    # Converting everything to torch tensors
    input_ids = torch.stack([torch.tensor(x["input_ids"]) for x in batch])
    attention_mask = torch.stack([torch.tensor(x["attention_mask"]) for x in batch])
    labels = torch.tensor([x["labels"] for x in batch])

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels
    }

In [ ]:
# Testing just a few transcripts
model_df_copy = model_df.copy()
model_df_copy.columns = ['text', 'label']
model_df_copy = model_df_copy[:10]
dataset_small = Dataset.from_pandas(model_df_copy)
dataset_small = dataset_small.remove_columns('__index_level_0__')

# Applying tokenizer and chunking
tokenized_dataset = dataset_small.map(
    lambda batch: batched_tokenize_and_chunk(batch, tokenizer=finbert_tokenizer),
    batched=True,
    remove_columns=dataset_small.column_names
)

tokenized_dataset

Map:   0%|          | 0/10 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 10
})

In [ ]:
# Creating a DataLoader just to check if things work
dataloader = DataLoader(
    tokenized_dataset,
    batch_size=4,
    shuffle=False,
    collate_fn=chunked_collate_fn
)

# Checking to see if everything works (just loading one sample)
for batch in dataloader:
    print(batch['input_ids'].shape)
    print(batch['attention_mask'].shape)
    print(batch['labels'].shape)
    break

torch.Size([4, 16, 512])
torch.Size([4, 16, 512])
torch.Size([4])


That seems correct, so let's move onto writing our custom module.

### Getting a CLS embedding extractor
I'll now define a class to just extract the CLS token embeddings from each chunk.

In [ ]:
# Defining a CLS embedding extractor wrapper around FinBERT
class FinBERTCLSExtractor(nn.Module):
    def __init__(self, base_model):
        super(FinBERTCLSExtractor, self).__init__()
        self.bert = base_model

    def forward(self, input_ids, attention_mask):

        # Just getting the input_ids and attention masks from the chunks
        batch_size, num_chunks, chunk_size = input_ids.shape
        input_ids = input_ids.view(-1, chunk_size)
        attention_mask = attention_mask.view(-1, chunk_size)

        # Disabling gradient calculation
        with torch.no_grad():

            # Getting the outputs and the first (CLS) token
            outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
            cls_embeddings = outputs.last_hidden_state[:, 0, :]

        cls_embeddings = cls_embeddings.view(batch_size, num_chunks, -1)
        return cls_embeddings

### Bidirectional LSTM layer
Now that we have our CLS token embeddings, we can feed them into a bidirectional LSTM layer to account for the order of the tokens. For example, an earnings call that starts off positive but ends more negative is likely to have a different overall sentiment from an earnings call that starts more negatively but ends more positively.

In [ ]:
# Defining the bidirectional LSTM layer
class FinBERTChunkedBiLSTMClassifier(nn.Module):
    def __init__(self, embedding_dim=768, hidden_dim=256, num_layers=1, dropout=0.3, num_classes=2):

        # Getting the LSTM layer
        super(FinBERTChunkedBiLSTMClassifier, self).__init__()
        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
            bidirectional=True
        )
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_dim * 2, num_classes)

    # Ultimately getting the cls embeddings in the dims: [batch_size, num_chunks, embedding_dim]
    def forward(self, cls_embeddings):
        lstm_out, _ = self.lstm(cls_embeddings)
        pooled_output = lstm_out[:, -1, :]
        pooled_output = self.dropout(pooled_output)
        logits = self.classifier(pooled_output)
        return logits

### Combining the models
Now I'll combine the LSTM and the embedding extractors together.

In [ ]:
# Combining the previous layers together
class FinBERTChunkedClassifier(nn.Module):
    def __init__(self, finbert_model, lstm_hidden_dim=256, num_layers=1, dropout=0.3, num_classes=2):
        super(FinBERTChunkedClassifier, self).__init__()
        self.cls_extractor = FinBERTCLSExtractor(finbert_model)
        self.lstm_classifier = FinBERTChunkedBiLSTMClassifier(
            embedding_dim=768,
            hidden_dim=lstm_hidden_dim,
            num_layers=num_layers,
            dropout=dropout,
            num_classes=num_classes
        )

    def forward(self, input_ids, attention_mask):
        cls_embeddings = self.cls_extractor(input_ids, attention_mask)
        logits = self.lstm_classifier(cls_embeddings)
        return logits

In [ ]:
# Defining a HuggingFace wrapper with the loss attached
class HFWrapperForChunkedClassifier(nn.Module):
    def __init__(self, model, loss_fn=None):
        super().__init__()
        self.model = model
        self.loss_fn = loss_fn or nn.CrossEntropyLoss()

    def forward(self, input_ids=None, attention_mask=None, labels=None):
        logits = self.model(input_ids, attention_mask)

        if labels is not None:
            loss = self.loss_fn(logits, labels)
            return {"loss": loss, "logits": logits}
        else:
            return {"logits": logits}

In [ ]:
# Defining a function for computing the metrics
metric = evaluate.load('accuracy')

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=1)
    return metric.compute(predictions=predictions, references=labels)

### Defining a fine-tuning classifier
Now that we've got everything we need, we can finally get our final classifier.

In [ ]:
# Defining fine-tuning classifer
def fine_tune_chunked_model(wrapped_model,
                            train_dataset,
                            val_dataset,
                            layers_to_train=['model.lstm_classifier.'],
                            batch_size=8,
                            num_epochs=2,
                            saved_output_dir="./tmp/temp_model_1_1"):

    # Freezing all model parameters except those explicitly listed
    for name, param in wrapped_model.named_parameters():
        if not any(x in name for x in layers_to_train):
            param.requires_grad = False
        else:
            param.requires_grad = True

    training_args = TrainingArguments(
        output_dir=saved_output_dir,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        num_train_epochs=num_epochs,
        eval_strategy="epoch",
        save_strategy="epoch",
        report_to="none",
        # load_best_model_at_end=True,
        metric_for_best_model="accuracy",
        greater_is_better=True
    )

    trainer = Trainer(
        model=wrapped_model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        data_collator=chunked_collate_fn,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
    )

    trainer.train()
    return trainer

In [ ]:
# Setting torch to high for better performance
torch.set_float32_matmul_precision('high')

### Testing fine-tuning
Let's perform a quick test with the smaller dataset to make sure things are working properly.

In [ ]:
# Performing a test with the datasets
split_dataset = tokenized_dataset.train_test_split(test_size=0.2, seed=42)
train_dataset = split_dataset["train"]
val_dataset = split_dataset["test"]

# Initiating the model and wrapper
chunked_model = FinBERTChunkedClassifier(finbert_model=finbert)
wrapped_model = HFWrapperForChunkedClassifier(chunked_model)

In [ ]:
# Confirming the parameters
for name, param in wrapped_model.named_parameters():
    print(name, param.shape)

model.cls_extractor.bert.embeddings.word_embeddings.weight torch.Size([30522, 768])
model.cls_extractor.bert.embeddings.position_embeddings.weight torch.Size([512, 768])
model.cls_extractor.bert.embeddings.token_type_embeddings.weight torch.Size([2, 768])
model.cls_extractor.bert.embeddings.LayerNorm.weight torch.Size([768])
model.cls_extractor.bert.embeddings.LayerNorm.bias torch.Size([768])
model.cls_extractor.bert.encoder.layer.0.attention.self.query.weight torch.Size([768, 768])
model.cls_extractor.bert.encoder.layer.0.attention.self.query.bias torch.Size([768])
model.cls_extractor.bert.encoder.layer.0.attention.self.key.weight torch.Size([768, 768])
model.cls_extractor.bert.encoder.layer.0.attention.self.key.bias torch.Size([768])
model.cls_extractor.bert.encoder.layer.0.attention.self.value.weight torch.Size([768, 768])
model.cls_extractor.bert.encoder.layer.0.attention.self.value.bias torch.Size([768])
model.cls_extractor.bert.encoder.layer.0.attention.output.dense.weight torch.

In [ ]:
# Testing fine-tuning
trainer = fine_tune_chunked_model(
    wrapped_model=wrapped_model,
    train_dataset=train_dataset,
    val_dataset=val_dataset,
    batch_size=8,
    num_epochs=2,
    saved_output_dir="./tmp/temp_model"
)

Using EarlyStoppingCallback without load_best_model_at_end=True. Once training is finished, the best model will not be loaded automatically.
/usr/local/lib/python3.11/dist-packages/torch/nn/modules/module.py:1750: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


Epoch,Training Loss,Validation Loss,Accuracy
1,No log,0.738886,0.000000
2,No log,0.724853,0.000000


/usr/local/lib/python3.11/dist-packages/torch/nn/modules/module.py:1750: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


While this was a small test, it shows that our model can work properly. Therefore, we'll move onto the larger test with the full dataset.

## Running full model on whole dataset
As mentioned, now that we've tested everything, let's put it altogether and streamline the model process.

In [ ]:
# Defining a function for computing the metrics
metric = evaluate.load('accuracy')

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=1)
    return metric.compute(predictions=predictions, references=labels)

# Setting torch to high for better performance
torch.set_float32_matmul_precision('high')

In [ ]:
# Defining all necessary functions

# Defining function to tokenize and chunk the batched text
def batched_tokenize_and_chunk(batch, tokenizer, chunk_size=CHUNK_SIZE, max_total_tokens=MAX_TOTAL_TOKENS):

    # Initializing input_ids and attention_mask batch lists
    input_ids_batch = []
    attention_mask_batch = []

    # Going through each text
    for text in batch['text']:

        # Tokenizing and truncating/padding
        tokens = tokenizer.encode(
            text,
            padding="max_length",
            truncation=True,
            max_length=max_total_tokens
        )
        # Breaking into chunks of 512 tokens and getting a list for the ids and masks
        chunks = [tokens[i:i + chunk_size] for i in range(0, len(tokens), chunk_size)]
        masks = [[1 if token != tokenizer.pad_token_id else 0 for token in chunk] for chunk in chunks]
        input_ids_batch.append(torch.tensor(chunks))
        attention_mask_batch.append(torch.tensor(masks))

    # Getting the output
    output = {
        "input_ids": input_ids_batch,
        "attention_mask": attention_mask_batch,
    }

    # Adding the label (if exists)
    if "label" in batch:
        output["labels"] = torch.tensor(batch["label"])

    return output

# Defining follow-up function to collate everything into torch tensors
def chunked_collate_fn(batch):

    # Converting everything to torch tensors
    input_ids = torch.stack([torch.tensor(x["input_ids"]) for x in batch])
    attention_mask = torch.stack([torch.tensor(x["attention_mask"]) for x in batch])
    labels = torch.tensor([x["labels"] for x in batch])

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels
    }

# Defining fine-tuning classifer
def fine_tune_chunked_model(wrapped_model,
                            train_dataset,
                            val_dataset,
                            layers_to_train=['model.lstm_classifier.'],
                            batch_size=8,
                            num_epochs=2,
                            saved_output_dir="./tmp/temp_model_1_1",
                            resume_from_checkpoint=False):

    # Freezing all model parameters except those explicitly listed
    for name, param in wrapped_model.named_parameters():
        if not any(x in name for x in layers_to_train):
            param.requires_grad = False
        else:
            param.requires_grad = True

    training_args = TrainingArguments(
        output_dir=saved_output_dir,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        num_train_epochs=num_epochs,
        eval_strategy="epoch",
        save_strategy="epoch",
        report_to="none",
        # load_best_model_at_end=True,
        metric_for_best_model="accuracy",
        greater_is_better=True
    )

    trainer = Trainer(
        model=wrapped_model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        data_collator=chunked_collate_fn,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
    )

    trainer.train(resume_from_checkpoint=resume_from_checkpoint)
    return trainer

In [ ]:
# Defining all of our class models

# Defining a CLS embedding extractor wrapper around FinBERT
class FinBERTCLSExtractor(nn.Module):
    def __init__(self, base_model):
        super(FinBERTCLSExtractor, self).__init__()
        self.bert = base_model

    def forward(self, input_ids, attention_mask):

        # Just getting the input_ids and attention masks from the chunks
        batch_size, num_chunks, chunk_size = input_ids.shape
        input_ids = input_ids.view(-1, chunk_size)
        attention_mask = attention_mask.view(-1, chunk_size)

        # Disabling gradient calculation
        with torch.no_grad():

            # Getting the outputs and the first (CLS) token
            outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
            cls_embeddings = outputs.last_hidden_state[:, 0, :]

        cls_embeddings = cls_embeddings.view(batch_size, num_chunks, -1)
        return cls_embeddings

# Defining the bidirectional LSTM layer
class FinBERTChunkedBiLSTMClassifier(nn.Module):
    def __init__(self, embedding_dim=768, hidden_dim=256, num_layers=1, dropout=0.3, num_classes=2):

        # Getting the LSTM layer
        super(FinBERTChunkedBiLSTMClassifier, self).__init__()
        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
            bidirectional=True
        )
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_dim * 2, num_classes)

    # Ultimately getting the cls embeddings in the dims: [batch_size, num_chunks, embedding_dim]
    def forward(self, cls_embeddings):
        lstm_out, _ = self.lstm(cls_embeddings)
        pooled_output = lstm_out[:, -1, :]
        pooled_output = self.dropout(pooled_output)
        logits = self.classifier(pooled_output)
        return logits

# Combining the previous layers together
class FinBERTChunkedClassifier(nn.Module):
    def __init__(self, finbert_model, lstm_hidden_dim=256, num_layers=1, dropout=0.3, num_classes=2):
        super(FinBERTChunkedClassifier, self).__init__()
        self.cls_extractor = FinBERTCLSExtractor(finbert_model)
        self.lstm_classifier = FinBERTChunkedBiLSTMClassifier(
            embedding_dim=768,
            hidden_dim=lstm_hidden_dim,
            num_layers=num_layers,
            dropout=dropout,
            num_classes=num_classes
        )

    def forward(self, input_ids, attention_mask):
        cls_embeddings = self.cls_extractor(input_ids, attention_mask)
        logits = self.lstm_classifier(cls_embeddings)
        return logits

# Defining a HuggingFace wrapper with the loss attached
class HFWrapperForChunkedClassifier(nn.Module):
    def __init__(self, model, loss_fn=None):
        super().__init__()
        self.model = model
        self.loss_fn = loss_fn or nn.CrossEntropyLoss()

    def forward(self, input_ids=None, attention_mask=None, labels=None):
        logits = self.model(input_ids, attention_mask)

        if labels is not None:
            loss = self.loss_fn(logits, labels)
            return {"loss": loss, "logits": logits}
        else:
            return {"logits": logits}

In [ ]:
# Defining max total tokens and chunk size
MAX_TOTAL_TOKENS = 8192
CHUNK_SIZE = 512

# Renaming the labels in model_df
model_df.columns = ['text', 'label']
dataset = Dataset.from_pandas(model_df)
dataset = dataset.remove_columns('__index_level_0__')

# Applying tokenizer and chunking
tokenized_dataset = dataset.map(
    lambda batch: batched_tokenize_and_chunk(batch, tokenizer=finbert_tokenizer),
    batched=True,
    remove_columns=dataset.column_names
)

tokenized_dataset

Map:   0%|          | 0/17233 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 17233
})

In [ ]:
# Splitting the datasets
split_dataset = tokenized_dataset.train_test_split(test_size=0.2, seed=42)
train_dataset = split_dataset["train"]
val_dataset = split_dataset["test"]

In [ ]:
# Initiating the model and wrapper
chunked_model = FinBERTChunkedClassifier(finbert_model=finbert)
wrapped_model = HFWrapperForChunkedClassifier(chunked_model)

# Testing fine-tuning
trainer = fine_tune_chunked_model(
    wrapped_model=wrapped_model,
    train_dataset=train_dataset,
    val_dataset=val_dataset,
    layers_to_train=['model.lstm_classifier.'],
    batch_size=8,
    num_epochs=5,
    saved_output_dir="./tmp/temp_model"
)

Using EarlyStoppingCallback without load_best_model_at_end=True. Once training is finished, the best model will not be loaded automatically.
/usr/local/lib/python3.11/dist-packages/torch/nn/modules/module.py:1750: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


Epoch,Training Loss,Validation Loss,Accuracy
1,0.694900,0.691509,0.533217
2,0.691400,0.690394,0.535828


/usr/local/lib/python3.11/dist-packages/torch/nn/modules/module.py:1750: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)
/usr/local/lib/python3.11/dist-packages/torch/nn/modules/module.py:1750: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


Epoch,Training Loss,Validation Loss,Accuracy
1,0.694900,0.691509,0.533217
2,0.691400,0.690394,0.535828


KeyboardInterrupt: 

My runtime disconnected, so I'll continue training from here:

In [ ]:
# Fine-tuning
trainer = fine_tune_chunked_model(
    wrapped_model=wrapped_model,
    train_dataset=train_dataset,
    val_dataset=val_dataset,
    layers_to_train=['model.lstm_classifier.'],
    batch_size=8,
    num_epochs=5,
    saved_output_dir="./tmp/temp_model",
    resume_from_checkpoint=True
)

Using EarlyStoppingCallback without load_best_model_at_end=True. Once training is finished, the best model will not be loaded automatically.
/usr/local/lib/python3.11/dist-packages/torch/nn/modules/module.py:1750: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


Epoch,Training Loss,Validation Loss,Accuracy
3,0.691400,0.690402,0.536408
4,0.689200,0.690509,0.535828
5,0.692600,0.690419,0.535828


/usr/local/lib/python3.11/dist-packages/torch/nn/modules/module.py:1750: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)
/usr/local/lib/python3.11/dist-packages/torch/nn/modules/module.py:1750: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


In [ ]:
# Getting actual predictions
predictions_output = trainer.predict(val_dataset)

# Extracting predicted class labels
y_pred = np.argmax(predictions_output.predictions, axis=1)
y_true = predictions_output.label_ids

# Generating the classification report
print(classification_report(y_true, y_pred, target_names=["Down", "Up"]))

              precision    recall  f1-score   support

        Down       0.31      0.00      0.01      1594
          Up       0.54      0.99      0.70      1853

    accuracy                           0.54      3447
   macro avg       0.42      0.50      0.35      3447
weighted avg       0.43      0.54      0.38      3447



Clearly, this model did not perform well, and it was still only comparable to the majority class baseline. Furthermore, we see our recurring problem with unbalanced recalls, which means this model is not learning very much. This goes against what we tried to account for, as the main goal of the chunking process was to overcome the recall issue.

However, what could be happening is that the model is training too quickly (with too granular of batch size). That might be what's causing the model to smooth out so quickly, and it's something that we can test with our next iteration.

The other concern is that the signal in the data is simply too faint, and even getting the right sentiment does not matter for prediction. Furthermore, getting the right sentiment also doesn't necessarily correspond to correct Up or Down predictions (if it did, someone would have long since found a way to utilize this to make money).

We'll see if these problems persist when testing the model on same day exits as entry.

## Testing model on same day exit as entry
As mentioned above, I'll test the model on its performance when the exit occurs the same day as entry.

This time, I'll also update a few more of the transformer layers, along with increasing the batch size to see if that makes a difference.

In [ ]:
# Getting the same day results (Close_0_dir)
model_df_20_same_day = full_data[['Text', 'Close_0_dir']].copy()
model_df_20_same_day.columns = ['text', 'label']
model_df_20_same_day['label'] = model_df_20_same_day['label'].astype(np.int64)

# Getting an updated dataset
dataset_exit_day_0 = Dataset.from_pandas(model_df_20_same_day)
dataset_exit_day_0 = dataset_exit_day_0.remove_columns('__index_level_0__')

# Applying tokenizer and chunking
tokenized_dataset_exit_day_0 = dataset_exit_day_0.map(
    lambda batch: batched_tokenize_and_chunk(batch, tokenizer=finbert_tokenizer),
    batched=True,
    remove_columns=dataset_exit_day_0.column_names
)

tokenized_dataset_exit_day_0

Map:   0%|          | 0/17233 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 17233
})

In [ ]:
# Splitting the datasets
split_dataset_ed0 = tokenized_dataset_exit_day_0.train_test_split(test_size=0.2, seed=42)
train_dataset_ed0 = split_dataset_ed0["train"]
val_dataset_ed0 = split_dataset_ed0["test"]

In [ ]:
# Initiating the model and wrapper
chunked_model_ed0 = FinBERTChunkedClassifier(finbert_model=finbert)
wrapped_model_ed0 = HFWrapperForChunkedClassifier(chunked_model_ed0)

# Testing fine-tuning
trainer_ed0 = fine_tune_chunked_model(
    wrapped_model=wrapped_model_ed0,
    train_dataset=train_dataset_ed0,
    val_dataset=val_dataset_ed0,
    layers_to_train=['model.cls_extractor.bert.encoder.layer.11.',
                     'model.cls_extractor.bert.pooler.', 'model.lstm_classifier.'],
    batch_size=32,
    num_epochs=10,
    saved_output_dir="./finbert/test_2_ed0",
    resume_from_checkpoint=False
)

Using EarlyStoppingCallback without load_best_model_at_end=True. Once training is finished, the best model will not be loaded automatically.
/usr/local/lib/python3.11/dist-packages/torch/nn/modules/module.py:1750: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


Epoch,Training Loss,Validation Loss,Accuracy
1,No log,0.697423,0.498695
2,0.695200,0.693707,0.498985
3,0.694200,0.693267,0.498404
4,0.695300,0.697038,0.499275
5,0.693700,0.693023,0.503626
6,0.693700,0.693275,0.505367
7,0.694000,0.693499,0.499275
8,0.694000,0.693675,0.499275
9,0.693600,0.693031,0.497534


/usr/local/lib/python3.11/dist-packages/torch/nn/modules/module.py:1750: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)
/usr/local/lib/python3.11/dist-packages/torch/nn/modules/module.py:1750: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)
/usr/local/lib/python3.11/dist-packages/torch/nn/modules/module.py:1750: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)
/usr/local/lib/python3.11/dist-packages/torch/nn/modules/module.py:1750: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)
/usr/local/lib/python3.11/dist-packa

In [ ]:
# Getting actual predictions
predictions_output_ed0 = trainer_ed0.predict(val_dataset_ed0)

# Extracting predicted class labels
y_pred_ed0 = np.argmax(predictions_output_ed0.predictions, axis=1)
y_true_ed0 = predictions_output_ed0.label_ids

# Generating the classification report
print(classification_report(y_true_ed0, y_pred_ed0, target_names=["Down", "Up"]))

/usr/local/lib/python3.11/dist-packages/torch/nn/modules/module.py:1750: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


              precision    recall  f1-score   support

        Down       0.50      0.95      0.65      1719
          Up       0.49      0.05      0.09      1728

    accuracy                           0.50      3447
   macro avg       0.49      0.50      0.37      3447
weighted avg       0.49      0.50      0.37      3447



In [ ]:
# Disconnecting the colab runtime once finished
# from google.colab import runtime
# runtime.unassign()

Unfortunately, we get similar results to before even after changing to same day exits, increasing the batch size, and increasing the number of layers to train. Again, the model does appear to be training, but it's not training very well.

However, before we continue with this test, let's test a hypothesis about sentiment itself. Specifically, what sentiment does base FinBERT predict for chunked earnings transcripts? If it's all the same (i.e., all positive), then running more FinBERT sentiment classifiers won't be helpful at all unless we fine-tune the whole model. If it's not, then we can try deciphering whether the poor results are due to the model architecture itself (or perhaps the way it's trained).

# Checking Overall Earnings Call Sentiment
Here, I'll test the hypothesis that my BERT models (particularly sentiment classifiers like FinBERT) are not performing well because earnings transcripts are inherently positive. I will do so by chunking up earnings calls into smaller sections - so they can fit into a FinBERT sentiment classifier - and check whether most of them are classified as positive. If so, it will confirm my hypothesis, and I can move onto testing decoder models to see if there's any improvements.

## Loading in FinBERT
Firstly, we'll load in FinBERT, but keeping the softmax so it can perform its 3-class classification.

In [ ]:
# Loading the FinBERT model and tokenizer from Hugging face
finbert_model_name = "ProsusAI/finbert"
finbert_tokenizer = AutoTokenizer.from_pretrained(finbert_model_name)
finbert_classification_model = BertForSequenceClassification.from_pretrained(finbert_model_name)

tokenizer_config.json:   0%|          | 0.00/252 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/758 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

In [ ]:
# Checking out the parameters
for name, param in finbert_classification_model.named_parameters():
    print(name, param.shape)

bert.embeddings.word_embeddings.weight torch.Size([30522, 768])
bert.embeddings.position_embeddings.weight torch.Size([512, 768])
bert.embeddings.token_type_embeddings.weight torch.Size([2, 768])
bert.embeddings.LayerNorm.weight torch.Size([768])
bert.embeddings.LayerNorm.bias torch.Size([768])
bert.encoder.layer.0.attention.self.query.weight torch.Size([768, 768])
bert.encoder.layer.0.attention.self.query.bias torch.Size([768])
bert.encoder.layer.0.attention.self.key.weight torch.Size([768, 768])
bert.encoder.layer.0.attention.self.key.bias torch.Size([768])
bert.encoder.layer.0.attention.self.value.weight torch.Size([768, 768])
bert.encoder.layer.0.attention.self.value.bias torch.Size([768])
bert.encoder.layer.0.attention.output.dense.weight torch.Size([768, 768])
bert.encoder.layer.0.attention.output.dense.bias torch.Size([768])
bert.encoder.layer.0.attention.output.LayerNorm.weight torch.Size([768])
bert.encoder.layer.0.attention.output.LayerNorm.bias torch.Size([768])
bert.encoder

## Chunking test
Testing the chunking of earnings call transcripts.

In [ ]:
# Getting the max length for FinBERT
MAX_LEN_FinBERT = 512

# Defining a function to tokenize the text
def tokenize_fn(texts, tokenizer):
    return tokenizer(
        texts,
        padding=False,
        truncation=False,
        max_length=MAX_LEN_FinBERT,
        return_tensors="pt"
    )

In [ ]:
# Getting the first text in the data
text_0 = tokenize_fn(model_df['Text'][0], finbert_tokenizer)

# Chunking
input_chunks = text_0['input_ids'][0].split(MAX_LEN_FinBERT)
attention_chunks = text_0['attention_mask'][0].split(MAX_LEN_FinBERT)

# Defining and storing a list of logits
logits_list = []
with torch.no_grad():
    for ids, mask in zip(input_chunks, attention_chunks):
        ids = ids.unsqueeze(0)
        mask = mask.unsqueeze(0)
        outputs = finbert_classification_model(input_ids=ids, attention_mask=mask)
        logits = outputs.logits
        logits_list.append(logits)

# Getting the average probabilities and the predicted class
probs = torch.softmax(torch.cat(logits_list, dim=0), dim=1)
avg_probs = probs.mean(dim=0)
pred_class = torch.argmax(avg_probs).item()

print('Average probabilities:', avg_probs)
print('Predicted class:', pred_class)

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

Average probabilities: tensor([0.5152, 0.0429, 0.4419])
Predicted class: 0


## FinBERT sentiment classifier for chunked transcripts
Getting sentiments from a portion of the earnings call transcripts using the FinBERT classifier.

*Note: I started testing it with all the earnings transcripts, but it would've taken ~10 more hours using the fastest GPU. Therefore, I'll just get a random 20% of the transcripts and evaluate their sentiments.

In [ ]:
# Picking only 20% of the rows in model_df
model_df_20 = model_df.sample(frac=0.20, random_state=42)
model_df_20 = model_df_20.reset_index(drop=True)

In [ ]:
# Defining a csv to save results to
finbert_senti_results_path = './finbert_senti_results.csv'

In [ ]:
# Defining function to get class predictions and average probabilities of text
def finbert_batch_chunk_classify(model_df, model, tokenizer, max_len=MAX_LEN_FinBERT):

    # Reading in saved results
    saved_senti_results = pd.read_csv(finbert_senti_results_path, sep='|', index_col='Unnamed: 0')

    # Defining a results df to store results in
    results = pd.DataFrame()

    # Disabling dropouts/batchnorm and gradients
    model.eval()
    with torch.no_grad():
        for idx, row in tqdm(model_df.iterrows(), total=len(model_df)):

            # Skipping if result already exists
            if idx in saved_senti_results.index:
                continue

            # Tokenizing full text without truncation
            text = row['Text']
            tokens = tokenize_fn(text, tokenizer)
            input_ids = tokens['input_ids'][0]
            attention_mask = tokens['attention_mask'][0]

            # Creating chunks of the FinBERT max length (512)
            input_chunks = input_ids.split(max_len)
            attention_chunks = attention_mask.split(max_len)

            # Getting the list of logits for each chunk
            chunk_logits = []
            for ids, mask in zip(input_chunks, attention_chunks):
                ids = ids.unsqueeze(0)
                mask = mask.unsqueeze(0)
                outputs = model(input_ids=ids, attention_mask=mask)
                logits = outputs.logits
                chunk_logits.append(logits)

            # Getting the average probabilities and the predicted class
            probs = torch.softmax(torch.cat(chunk_logits, dim=0), dim=1)
            avg_probs = probs.mean(dim=0)
            avg_probs_np = avg_probs.numpy()
            pred_class = torch.argmax(avg_probs).item()

            # Getting the first chunk prediction
            first_chunk_class = torch.argmax(probs[0]).item()

            # Adding to the results df
            results['avg_prob_neg'] = [avg_probs_np[0]]
            results['avg_prob_neu'] = [avg_probs_np[1]]
            results['avg_prob_pos'] = [avg_probs_np[2]]
            results['pred_class'] = [pred_class]
            results['first_chunk_class'] = [first_chunk_class]

            # Adding to the saved sentiment results and saving it
            saved_senti_results = pd.concat([saved_senti_results, results])
            saved_senti_results = saved_senti_results.reset_index(drop=True)
            saved_senti_results.to_csv(finbert_senti_results_path, sep='|')

    return results

In [ ]:
# Getting the chunk results
chunk_results = finbert_batch_chunk_classify(model_df_20, finbert_classification_model, finbert_tokenizer, MAX_LEN_FinBERT)

100%|██████████| 3447/3447 [1:10:33<00:00,  1.23s/it]


In [ ]:
# Reading the saved csv
tscrpt_senti_results = pd.read_csv(finbert_senti_results_path, sep='|', index_col='Unnamed: 0')

tscrpt_senti_results

,avg_prob_neg,avg_prob_neu,avg_prob_pos,pred_class,first_chunk_class
0,0.613876,0.060406,0.325719,0,2
1,0.458309,0.138161,0.403530,0,2
2,0.493030,0.087365,0.419605,0,2
3,0.753564,0.027896,0.218540,0,0
4,0.669266,0.057126,0.273608,0,2
...,...,...,...,...,...
3442,0.576985,0.052102,0.370913,0,2
3443,0.393574,0.173594,0.432832,2,2
3444,0.705405,0.074641,0.219954,0,2
3445,0.588332,0.020670,0.390998,0,2


***Note**: I explain this more below, but I found out afterward that my labels were incorrect. A label of 0 actually corresponds with positive, 1 with negative, and 2 with netural.

To save time and compute, I won't re-run this code, but this will need to be kept in mind for future analyses.

In [ ]:
# Getting some statistics on the chunks
display(tscrpt_senti_results['pred_class'].value_counts())
display(tscrpt_senti_results['first_chunk_class'].value_counts())

,count
pred_class,
0,2021
2,1421
1,5


,count
first_chunk_class,
2,2745
0,693
1,9


After double checking the FinBERT model card, I realize I made a mistake with the labeling. Specifically, I assumed {0: "negative", 1: "neutral", 2: "positive}. However, the FinBERT documentation actually reads "id2label": {"0": "positive", "1": "negative", "2": "neutral"}.

The good news is that this doesn't actually affect the sentiment classifier, and we can keep what we have (with the understanding that the labels are incorrect).

With that being said, we do see some striking results. Notably, when averaging the sentiment across chunks, it appears that the vast majority of transcripts are positive (0) or neutral (2). This provides strong reasoning as to why our previous models kept predicting the "Up" class, as it would've taken an inordinate amount of adjustments to tune the model to predict the "Down" class.

We also see why only using the first chunk did not work well, as the majority of predictions were neutral (which makes sense, as those are generally just the opening remarks).

With this in mind, we can likely conclude that trying to train more with FinBERT models won't yield any better results, as the idea of using FinBERT to distinguish between positive and negative sentiment doesn't seem to work in the first place. Thus, let's try moving to decoder models and see if we can find any better results.